# Render the Techpe Trek reels on Colab

Renders all 4 parts of the GitHub Trending series (`gh1`..`gh4`, Part 1 with the corrected "PART 1/4" badge) from the `motion-as-code` project, sharp (2x), 1080x1920, into `MyDrive/techpe-trek-renders/`.

**Before you start:** Runtime > Change runtime type > **T4 GPU** (or better). Upload `motion-as-code-colab.zip` (made on your PC) to the top of your Google Drive (`MyDrive`).

Voices and sound mixes are already inside the zip, so no API keys are needed here.

In [ ]:
# 1. Check the GPU
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!nproc; free -g | head -2

In [ ]:
# 2. Mount Drive and unpack the project to /content/motion-as-code
from google.colab import drive
drive.mount('/content/drive')
!rm -rf /content/motion-as-code && unzip -q /content/drive/MyDrive/motion-as-code-colab.zip -d /content/ && ls /content/motion-as-code

In [ ]:
# 3. Install ffmpeg, Google Chrome (for WebGL on the GPU), Vulkan loader, and bun
!apt-get -qq update && apt-get -qq install -y ffmpeg libvulkan1 vulkan-tools > /dev/null
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb && apt-get -qq install -y ./google-chrome-stable_current_amd64.deb > /dev/null
!curl -fsSL https://bun.sh/install | bash > /dev/null
import os
os.environ['PATH'] = '/root/.bun/bin:' + os.environ['PATH']
!google-chrome --version; bun --version; ffmpeg -version | head -1

In [ ]:
# 4. Install the app's packages
%cd /content/motion-as-code/app
!bun install

In [ ]:
# 5. Is Chrome's WebGL on the GPU? Look for 'WebGL: Hardware accelerated' and the NVIDIA renderer.
#    If it says Software / SwiftShader, try the other flag set in step 6 (CHROME_GPU_ARGS).
check = r'''
import { chromium } from 'playwright-core';
const b = await chromium.launch({ channel: 'chrome', headless: true, args: (process.env.CHROME_GPU_ARGS ?? '--use-angle=vulkan --enable-features=Vulkan --disable-vulkan-surface').split(' ').concat(['--no-sandbox', '--ignore-gpu-blocklist']) });
const p = await b.newPage(); await p.goto('chrome://gpu'); await p.waitForTimeout(2000);
const t = await p.evaluate(() => document.body.innerText);
console.log(t.split('\n').filter((l) => /WebGL|Canvas|GL_RENDERER|ANGLE|Vulkan/.test(l)).slice(0, 14).join('\n'));
await b.close();
'''
open('gpu_check.ts', 'w').write(check)
!bun gpu_check.ts

In [ ]:
# 6. (Only if step 5 showed software rendering) try EGL instead of Vulkan, then re-run step 5
# os.environ['CHROME_GPU_ARGS'] = '--use-gl=angle --use-angle=gl-egl'
# Quick test: one still from Part 2, saved to out/colab_test; look at it in the file browser
!bun scripts/render.ts stills --variant gh2 --scale 2 --t 12 --out ../out/colab_test 2>&1 | grep -v 'GPU stall' | tail -3

In [ ]:
# 7. Render the parts one after another (each in 15 s segments that retry if they drop).
#    Each finished video is copied to Drive at once, so a disconnect never loses a finished part.
#    If the session drops: re-run steps 2-4, then delete the parts already in Drive from this list.
parts = {'gh1': 1, 'gh2': 2, 'gh3': 3, 'gh4': 4}
!mkdir -p /content/drive/MyDrive/techpe-trek-renders
for v, n in parts.items():
    !bash scripts/render_reel.sh {v} ../out/github-trending-part{n}.mp4 && cp ../out/github-trending-part{n}.mp4 /content/drive/MyDrive/techpe-trek-renders/
!ls -la /content/drive/MyDrive/techpe-trek-renders

In [ ]:
# 8. Upload check (H.264, 1080x1920, yuv420p, bt709, AAC) and copy the videos to Drive
!for f in ../out/github-trending-part*.mp4; do echo $f; ffprobe -v error -show_entries stream=codec_name,width,height,pix_fmt,color_range:format=duration -of compact "$f"; done
!mkdir -p /content/drive/MyDrive/techpe-trek-renders && cp ../out/github-trending-part*.mp4 /content/drive/MyDrive/techpe-trek-renders/ && ls /content/drive/MyDrive/techpe-trek-renders